# Learned-vs-gated sign head benchmark — heatmaps

Two metrics (relative energy error, exact fidelity `1-F`) across the
`docs/signhead_benchmark_plan.md` field grid (2x3 doubled semion,
`hx in {0.4, 0.8, 1.2}`, `hz in {0, 0.2, 0.4}`) for the three sign arms:

| arm | wavefunction |
|---|---|
| **M** (`cnnqM`) | `psi = A(sigma) * tanh(m_theta(eps, x))`, `m_theta` random init |
| **M-pre** (`cnnqMp`) | same architecture, `m_theta` supervised-pretrained to the exact ED signs first (`scripts/pretrain_sign_mlp.py`) |
| **T** (`cnnqT`) | `psi = a A_triv(sigma) + (-1)^s(sigma) A_top(sigma)`, `a` a signed real mix scalar (init `--mix_init 0.05`; `a = 0` is the head-only arm) |

Data comes from `scripts/signbench_summary.py`'s JSON output (rel-err vs the
exact `E0`, `1-F` vs the exact trained-state fidelity from
`scripts/nqs_fidelity.py`, and each arm's ceiling: `T_gate` for T (plan
Sec 7 -- the ED weight on the wrong global sign in the head-sector the
positive trunks cannot flip, min over the two sectors; falls back to the mwpm
head's `1 - F_s`, flagged `(1-Fs)`, on signfid JSONs predating it), exactly 0
for M/M-pre since `(eps, x)` determines `sigma` bijectively). Each `1-F` cell
is annotated with its ceiling. Missing cells (no checkpoint yet) are grey /
"missing"; an EXACT 0 (perfect) is drawn at the colour scale's floor and
annotated `0 (exact)` so it is never confused with a missing cell; a NaN
training tail is flagged in its cell text. Column titles carry each arm's
parameter count and sign prior (random / ED-pretrained warm start with its
warm-start `1-F_s` / head-only init with `mix_init`) -- the fairness
bookkeeping of `scripts/signbench_summary.py`'s `n_params` / `prior` columns.
Full numeric table incl. seeds and verdicts: `python scripts/signbench_summary.py`.

Row 2 of each figure is a **placeholder for the 3D fermionic-toric-code
panels** (peer repo `toric-code-nqs-fsign`, same plan Sec 5) — see the
schema cell near the bottom for what that repo should emit so this notebook
picks it up automatically.

In [ ]:
import os, sys, json, pathlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# house plot style (shared with the toric-code-nqs repo)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
FIGDIR = "figures"

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'results' / 'nqs').is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError('run this notebook from inside the 2D-TC repo '
                           '(results/nqs not found walking up from the kernel cwd)')
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

SUMMARY_JSON = "results/diagnostics/signbench_summary.json"
THREED_JSON = "results/diagnostics/signbench_3d.json"   # optional, peer repo

In [ ]:
# Load (or, on a fresh checkout with no cached summary yet, regenerate) the
# 2D grading table. The canonical way to produce SUMMARY_JSON is the CLI:
#   python scripts/signbench_summary.py --out results/diagnostics/signbench_summary.json
# -- this cell falls back to calling the same functions directly so the
# notebook is still runnable right after `cluster.sh fetch` with no extra step.
if not os.path.exists(SUMMARY_JSON):
    from scripts.signbench_summary import (load_refs, load_runs, load_fidelity,
                                            build_records, verdicts, ARMS)
    SIZE = "2x3"
    POINTS = [(round(hx, 6), round(hz, 6))
              for hx in (0.4, 0.8, 1.2) for hz in (0.0, 0.2, 0.4)]
    refs = load_refs()
    runs = load_runs("results/nqs")
    fid = load_fidelity("results/diagnostics")
    records = build_records(SIZE, POINTS, list(ARMS), runs, fid, refs, 20)
    summary = {"Lx": 2, "Ly": 3, "size": SIZE, "points": POINTS, "arms": list(ARMS),
              "tail": 20, "records": records, "verdicts": verdicts(records)}
    os.makedirs(os.path.dirname(SUMMARY_JSON), exist_ok=True)
    with open(SUMMARY_JSON, "w") as f:
        json.dump(summary, f, indent=1)
    print(f"(regenerated {SUMMARY_JSON})")

summary = json.load(open(SUMMARY_JSON))
records_2d = summary["records"]
print(f"2D: {len(records_2d)} records loaded from {SUMMARY_JSON} "
      f"({summary['size']}, {len(summary['verdicts'])} verdicts)")

records_3d = None
if os.path.exists(THREED_JSON):
    d3 = json.load(open(THREED_JSON))
    records_3d = d3["records"]
    print(f"3D: {len(records_3d)} records loaded from {THREED_JSON} ({d3.get('size')})")
else:
    print(f"3D: no {THREED_JSON} yet -- row 2 below stays a placeholder")

## Figure: rel-err and `1-F`, 3x3 field grid x 3 arms

One figure per metric; row 1 = this repo's 2x3 doubled semion, row 2 = the
peer repo's 3D fermionic-toric-code grid once `signbench_3d.json` exists
(placeholder otherwise). Columns = arms M / M-pre / T. Shared `LogNorm`
colour scale per metric (across both rows, so the two lattices are directly
comparable).

In [ ]:
ARM_LABEL = {'cnnqM': 'M', 'cnnqMp': 'M-pre', 'cnnqT': 'T'}


def cell_grid(records):
    '''records -> {(hx, hz, arm): record} using seed 0 only (seed-1
    replicas feed the verdict table, not the headline heatmap number).'''
    return {(r['hx'], r['hz'], r['arm']): r for r in records if r.get('seed', 0) == 0}


def axis_values(records):
    hx = sorted({r['hx'] for r in records})
    hz = sorted({r['hz'] for r in records})
    return hx, hz


def ceiling_label(r):
    '''per-cell ceiling annotation: T -> T_gate (plan Sec 7) or the flagged
    1-F_s fallback; M / M-pre -> exact 0.'''
    c = r.get('ceiling')
    if c is None:
        return 'ceil ?'
    kind = r.get('ceiling_kind') or ''
    flag = ' (1-Fs)' if kind == '1-Fs' else ''
    return ('ceil 0' if c == 0 else f'ceil {c:.0e}') + flag


def short_prior(r):
    '''compact prior for a column title (full string lives in r["prior"]).'''
    arm = r.get('arm')
    if arm == 'cnnqM':
        return 'random sign init'
    if arm == 'cnnqMp':
        w = r.get('warm_1mFs')
        return 'ED-pretrained' + (f' (warm 1-Fs {w:.0e})' if w is not None else '')
    if arm == 'cnnqT':
        p = r.get('prior') or ''
        return 'head-only init' + (' ' + p[p.index('(mix'):] if '(mix' in p else '')
    return r.get('prior') or ''


def arm_title(arm, records):
    '''"<label> | <n_params> params | <prior>" from the arm's seed-0 records
    (param count is per arm -- identical across the grid -- and the M-pre
    warm-start 1-F_s is per cell, so it is annotated in the cell instead).'''
    recs = [r for r in records if r.get('arm') == arm and not r.get('missing')]
    label = ARM_LABEL.get(arm, arm)
    if not recs:
        return label
    n = {r.get('n_params') for r in recs} - {None}
    n_txt = (f"{n.pop()} params" if len(n) == 1 else
             f"{min(n)}-{max(n)} params" if n else '? params')
    pri = short_prior(recs[0])
    if arm == 'cnnqMp':
        pri = 'ED-pretrained (warm 1-Fs per cell)'
    return f"{label} | {n_txt}\n{pri}"


def draw_row(fig, axes_row, records, arms, metric_key, norm, row_label,
             annotate_ceiling=False):
    '''Fill one row of (len(arms)) axes with the metric heatmap; a row with
    no records is drawn as a grey "no data yet" placeholder instead.'''
    if not records:
        for ax, arm in zip(axes_row, arms):
            ax.grid(False)
            ax.set_facecolor('0.93')
            ax.text(0.5, 0.5, 'no data yet', ha='center', va='center',
                    fontsize=11, style='italic', color='0.5',
                    transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(ARM_LABEL.get(arm, arm))
        return None

    cell = cell_grid(records)
    hx_vals, hz_vals = axis_values(records)
    im = None
    for ax, arm in zip(axes_row, arms):
        ax.grid(False)
        M = np.full((len(hz_vals), len(hx_vals)), np.nan)
        for j, hz in enumerate(hz_vals):
            for i, hx in enumerate(hx_vals):
                r = cell.get((hx, hz, arm))
                v = r.get(metric_key) if r else None
                if v is not None and v >= 0:
                    # exact 0 (perfect) is floored at the LogNorm vmin so it
                    # renders as the darkest cell -- distinct from missing
                    # (grey) -- and gets annotated below
                    M[j, i] = max(v, norm.vmin)
        im = ax.imshow(np.ma.masked_invalid(M), origin='lower',
                       extent=(-0.5, len(hx_vals) - 0.5, -0.5, len(hz_vals) - 0.5),
                       cmap='Blues', norm=norm, interpolation='nearest')
        ax.set_facecolor('0.88')
        for j, hz in enumerate(hz_vals):
            for i, hx in enumerate(hx_vals):
                r = cell.get((hx, hz, arm))
                if r is None or r.get('missing'):
                    ax.text(i, j, 'missing', ha='center', va='center',
                            fontsize=7, style='italic', color='0.35')
                    continue
                v = r.get(metric_key)
                if v is None:
                    ax.text(i, j, '?', ha='center', va='center',
                            fontsize=9, color='0.3')
                    continue
                dark = norm(max(v, norm.vmin)) > 0.6
                txt = ('0 (exact)' if v == 0 else f'{v:.1e}') \
                    + (' NaN' if r.get('nan_tail') else '')
                if annotate_ceiling:
                    txt += '\n' + ceiling_label(r)
                    if r.get('arm') == 'cnnqMp' and r.get('warm_1mFs') is not None:
                        txt += f"\nwarm {r['warm_1mFs']:.0e}"
                ax.text(i, j, txt, ha='center', va='center', fontsize=6.4,
                       color='white' if dark else '#1a1a1a')
        ax.set_xticks(range(len(hx_vals)))
        ax.set_xticklabels([f'{x:g}' for x in hx_vals])
        ax.set_yticks(range(len(hz_vals)))
        ax.set_yticklabels([f'{z:g}' for z in hz_vals])
        ax.set_title(arm_title(arm, records), fontsize=8.5)
    axes_row[0].set_ylabel(f'{row_label}\n' + r'$h_z$')
    return im


def plot_metric(metric_key, metric_label, records_2d, records_3d, arms,
                annotate_ceiling=False):
    pos = [r[metric_key] for recs in (records_2d, records_3d or [])
           for r in recs if r.get(metric_key) not in (None,) and r[metric_key] > 0]
    # exact zeros are floored at vmin (annotated '0 (exact)'), so the scale
    # is set by the strictly positive values only
    norm = LogNorm(vmin=min(pos), vmax=max(pos)) if pos else LogNorm(vmin=1e-6, vmax=1.0)

    fig, axes = plt.subplots(2, len(arms), figsize=(3.8 * len(arms), 6.8),
                             constrained_layout=True)
    im = draw_row(fig, axes[0], records_2d, arms, metric_key, norm, '2x3 (DS)',
                  annotate_ceiling=annotate_ceiling)
    draw_row(fig, axes[1], records_3d, arms, metric_key, norm, '3D fTC',
             annotate_ceiling=annotate_ceiling)
    for ax in axes[-1]:
        ax.set_xlabel('$h_x$')
    if im is not None:
        fig.colorbar(im, ax=list(axes.ravel()), shrink=0.8, pad=0.02, label=metric_label)
    fig.suptitle(f'Sign-head benchmark: {metric_label}')
    priors = {ARM_LABEL.get(r['arm'], r['arm']): r['prior']
              for r in records_2d if r.get('prior') and r.get('seed', 0) == 0}
    cap = ('exact 0 = floor of the colour scale (annotated); grey = missing. '
           'Priors: ' + '; '.join(f'{k}: {v}' for k, v in priors.items())
           if priors else
           'exact 0 = floor of the colour scale (annotated); grey = missing.')
    fig.text(0.0, -0.02, cap, ha='left', va='top', fontsize=7.5, wrap=True)
    return fig


ARMS_HERE = summary['arms']
fig_rel = plot_metric('rel_err', r'rel. energy error $|E-E_0|/|E_0|$',
                      records_2d, records_3d, ARMS_HERE)
# fig_rel.savefig(f"{FIGDIR}/signbench_relerr.png", dpi=300, bbox_inches="tight")

In [ ]:
fig_f = plot_metric('one_minus_F', r'$1-F$ (exact trained-state fidelity)',
                    records_2d, records_3d, ARMS_HERE, annotate_ceiling=True)
# fig_f.savefig(f"{FIGDIR}/signbench_1minusF.png", dpi=300, bbox_inches="tight")

## Verdicts (plan Sec 6)

An arm "wins" a cell over another if its `1-F` is >= 3x lower on BOTH seed 0
and a seed-1 replica at the same point (`jobs/nersc_signbench.sh`'s
`SEED=1` convention, jobid suffix `_s1`). Only printed once two seeds with
`1-F` both exist for a pair of arms at a point -- empty until stage f
(seed replicas) is run.

In [ ]:
if summary['verdicts']:
    for v in summary['verdicts']:
        who = v['winner'] or 'no winner (< 3x on both seeds)'
        print(f"({v['hx']:g},{v['hz']:g}) {ARM_LABEL.get(v['arm_a'], v['arm_a'])} vs "
              f"{ARM_LABEL.get(v['arm_b'], v['arm_b'])}: {who}  "
              f"[1-F seed0: {v['one_minus_F_a_seed0']:.2e} vs {v['one_minus_F_b_seed0']:.2e}, "
              f"seed1: {v['one_minus_F_a_seed1']:.2e} vs {v['one_minus_F_b_seed1']:.2e}]")
else:
    print('no verdicts yet (need seed-1 replicas with 1-F on both arms at a point)')

## 3D panel data contract

`THREED_JSON = "results/diagnostics/signbench_3d.json"` — when the peer repo
(`toric-code-nqs-fsign`) produces this file, row 2 of both figures above
fills in automatically on the next re-run (no notebook changes needed). It
must carry the exact schema `scripts/signbench_summary.py` writes:

```json
{
  "Lx": <int|null>, "Ly": <int|null>, "size": "<label, e.g. '2x2x3' or 'L2'>",
  "points": [[hx, hz], ...],
  "arms": ["<arm tokens, e.g. the 3D M/M-pre/T equivalents>"],
  "tail": <int>,
  "records": [
    {"size": "<label>", "hx": <float>, "hz": <float>, "arm": "<token>",
     "seed": <int>, "missing": <bool>,
     "E_tail": <float|null>, "E_tail_std": <float|null>,
     "Vscore_tail": <float|null>, "nan_tail": <bool>,
     "E0": <float|null>, "rel_err": <float|null>,
     "F": <float|null>, "F_trunk": <float|null>, "one_minus_F": <float|null>,
     "ceiling": <float|null>, "ceiling_kind": <"T_gate"|"1-Fs"|"exact-0"|null>,
     "mix": <float|null>, "n_params": <int|null>, "prior": <str|null>,
     "warm_1mFs": <float|null>}
    , ...
  ],
  "verdicts": [ ... same shape as scripts/signbench_summary.verdicts() ... ]
}
```

`size` is just a display label (this notebook never parses it beyond using
it as a row caption); `arms` should list exactly `len(ARMS_HERE)` tokens IN
THE SAME M / M-pre / T order so the columns line up, and `ARM_LABEL` above
may need new entries if the peer repo's tokens differ from `cnnqM` /
`cnnqMp` / `cnnqT`. `hx`/`hz` are read as plain floats (rounded to 6 dp for
matching, same as `scripts/signbench_summary.py`); a 3D grid does not need
to share the exact same 9 values as the 2D one -- `axis_values()` derives
its own axis ticks per row from whatever `records_3d` contains.